# 02 — Model Risiko Sesi Terjadwal Kosong (SS1-14)

Notebook ini menjalankan seluruh alur model ML klasifikasi biner yang
memprediksi apakah sesi terjadwal berisiko tidak dipakai.

## Alur
1. Muat data dari database
2. Bangun fitur (hanya informasi sebelum sesi)
3. Pembagian kronologis
4. Baseline
5. Latih model (Decision Tree dan Logistic Regression)
6. Evaluasi validasi dan analisis kesalahan
7. Tabel overfitting
8. **Evaluasi akhir (set test)** — sel terpisah di akhir

In [ ]:
import sys
import os

# Pastikan ai-service di path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 150)

from app.ml.konstanta import (
    RANDOM_STATE,
    TANGGAL_LATIH_MULAI, TANGGAL_LATIH_SELESAI,
    TANGGAL_VALIDASI_MULAI, TANGGAL_VALIDASI_SELESAI,
    TANGGAL_UJI_MULAI, TANGGAL_UJI_SELESAI,
)
from app.ml.fitur import bangun_fitur, kolom_fitur, KOLOM_KATEGORIK, KOLOM_NUMERIK
from app.ml.pembagian import bagi_kronologis, verifikasi_pembagian
from app.ml.model import latih_dan_pilih, simpan_model
from app.ml.evaluasi import (
    baseline_mayoritas, baseline_rata_historis_ruang,
    baseline_rata_historis_jenis_hari,
    hitung_metrik, hitung_rata_per_ruang, hitung_rata_per_jenis_hari,
    cetak_metrik, bootstrap_ci, cetak_irisan_uji,
    analisis_kesalahan, pentingnya_fitur_tree, koefisien_logreg,
)

print(f'Random state: {RANDOM_STATE}')
print(f'numpy: {np.__version__}')
print(f'pandas: {pd.__version__}')

## 1. Muat data dari database

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from dotenv import load_dotenv

load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

db_user = os.getenv('DB_USERNAME', 'root')
db_password = os.getenv('DB_PASSWORD', '')
db_host = os.getenv('DB_HOST', '127.0.0.1')
db_port = os.getenv('DB_PORT', '3306')

# Set dev
url_dev = f'mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/smart_space_ai_dev' if db_password else f'mysql+pymysql://{db_user}@{db_host}:{db_port}/smart_space_ai_dev'
engine_dev = create_engine(url_dev, pool_pre_ping=True)
Session_dev = sessionmaker(bind=engine_dev)
session_dev = Session_dev()

from app.ml.dataset import muat_sesi_terjadwal, muat_calendar_exceptions

df_sesi = muat_sesi_terjadwal(session_dev)
cal_exc = muat_calendar_exceptions(session_dev)

print(f'Jumlah sesi terjadwal: {len(df_sesi)}')
print(f'Distribusi label:\n{df_sesi["label"].value_counts()}')
print(f'\nJumlah calendar_exceptions: {len(cal_exc)}')
df_sesi.head()

## 2. Bangun fitur

In [ ]:
df_fitur = bangun_fitur(df_sesi, cal_exc)
print(f'Jumlah fitur: {len(kolom_fitur())}')
print(f'Kolom: {list(df_fitur.columns)}')
df_fitur.head(10)

## 3. Pembagian kronologis

In [ ]:
df_latih, df_val, df_uji_dev = bagi_kronologis(df_fitur)
verifikasi_pembagian(df_latih, df_val, df_uji_dev)
print('\n✓ Pembagian kronologis terverifikasi (tidak tumpang tindih, berurutan)')

## 4. Baseline

In [ ]:
fitur = kolom_fitur()
fitur_ada = [f for f in fitur if f in df_latih.columns]

X_latih = df_latih[fitur_ada]
y_latih = df_latih['label'].values
X_val = df_val[fitur_ada]
y_val = df_val['label'].values

# Baseline 1: Mayoritas
m_maj = baseline_mayoritas(y_val)
cetak_metrik('Baseline: Mayoritas', m_maj)

# Baseline 2: Rata-rata historis per ruang
rata_ruang = hitung_rata_per_ruang(df_latih)
m_ruang = baseline_rata_historis_ruang(y_val, df_val['room_id'].values, rata_ruang)
cetak_metrik('Baseline: Rata-rata historis per ruang', m_ruang)

# Baseline 3: Rata-rata historis per jenis hari
rata_jenis = hitung_rata_per_jenis_hari(df_latih)
m_jenis = baseline_rata_historis_jenis_hari(y_val, df_val['jenis_hari'].values, rata_jenis)
cetak_metrik('Baseline: Rata-rata historis per jenis hari', m_jenis)

## 5. Latih model

In [ ]:
hasil = latih_dan_pilih(X_latih, y_latih, X_val, y_val)

## 6. Evaluasi validasi

In [ ]:
for nama_model, key in [('Decision Tree', 'tree'), ('Logistic Regression', 'logreg')]:
    pipe = hasil[key]['pipeline']
    ambang = hasil[key]['ambang']
    y_proba = pipe.predict_proba(X_val)[:, 1]
    
    # Ambang optimal
    y_pred_opt = (y_proba >= ambang).astype(int)
    m_opt = hitung_metrik(y_val, y_pred_opt, y_proba)
    cetak_metrik(f'{nama_model} (ambang={ambang})', m_opt)
    
    # Ambang 0.5
    y_pred_05 = (y_proba >= 0.5).astype(int)
    m_05 = hitung_metrik(y_val, y_pred_05, y_proba)
    cetak_metrik(f'{nama_model} (ambang=0.5)', m_05)

## 7. Tabel overfitting

In [ ]:
print('=== Tabel Overfitting: Decision Tree ===')
df_ov_dt = pd.DataFrame(hasil['tree']['tabel_overfitting'])
print(df_ov_dt.to_string(index=False))

print('\n=== Tabel Overfitting: Logistic Regression ===')
df_ov_lr = pd.DataFrame(hasil['logreg']['tabel_overfitting'])
print(df_ov_lr.to_string(index=False))

## 8. Analisis kesalahan

In [ ]:
# Analisis kesalahan Logistic Regression pada validasi
pipe_lr = hasil['logreg']['pipeline']
ambang_lr = hasil['logreg']['ambang']
y_proba_lr = pipe_lr.predict_proba(X_val)[:, 1]
y_pred_lr = (y_proba_lr >= ambang_lr).astype(int)

err = analisis_kesalahan(df_val, y_val, y_pred_lr)
for key_err, df_err in err.items():
    print(f'\n{key_err}:')
    print(df_err.to_string(index=False))

In [ ]:
# Feature importance Decision Tree
print('=== Feature Importance (Decision Tree) ===')
fi = pentingnya_fitur_tree(hasil['tree']['pipeline'], fitur_ada)
print(fi.head(15).to_string(index=False))

print('\n=== Koefisien (Logistic Regression) ===')
coef = koefisien_logreg(hasil['logreg']['pipeline'], fitur_ada)
print(coef.to_string(index=False))

## 9. Keterbatasan

- **Label berasal dari data simulasi dan sistem pakar.** Model bisa hanya mempelajari aturan generator; performanya tidak mencerminkan dunia nyata.
- **Jenis hari dominan.** Mayoritas data bertipe 'biasa', sehingga distribusi kelas pada ujian dan kegiatan_khusus mungkin terlalu kecil.
- **Sampel positif kecil pada irisan.** Masa ujian, kegiatan khusus, dan pekan libur mungkin punya kurang dari 20 sampel positif, membuat metrik tidak stabil.
- **Fitur riwayat.** `kosong_rate_ruang_sebelumnya` bergantung pada label sistem pakar yang juga berasal dari simulator.

Model ini bersifat *proof of concept*. Untuk penggunaan nyata, diperlukan data okupansi dari sensor sesungguhnya.

---

## ⚠ EVALUASI AKHIR (Set Test)

**PERINGATAN: Sel di bawah ini hanya boleh dieksekusi SEKALI.**
Jangan mengubah model, hiperparameter, atau ambang keputusan setelah melihat hasil.

Jalankan sel berikut hanya saat evaluasi akhir diperlukan.

In [ ]:
# === EVALUASI AKHIR — HANYA SEKALI ===
# Uncomment dan jalankan sel ini saat siap evaluasi akhir.

# url_test = f'mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/smart_space_ai_test' if db_password else f'mysql+pymysql://{db_user}@{db_host}:{db_port}/smart_space_ai_test'
# engine_test = create_engine(url_test, pool_pre_ping=True)
# Session_test = sessionmaker(bind=engine_test)
# session_test = Session_test()
#
# df_sesi_test = muat_sesi_terjadwal(session_test)
# cal_exc_test = muat_calendar_exceptions(session_test)
# df_fitur_test = bangun_fitur(df_sesi_test, cal_exc_test)
#
# from datetime import date
# tanggal_test = df_fitur_test['tanggal'].apply(lambda d: d if isinstance(d, date) else date.fromisoformat(str(d)))
# mask_uji = (tanggal_test >= TANGGAL_UJI_MULAI) & (tanggal_test <= TANGGAL_UJI_SELESAI)
# df_uji_test = df_fitur_test.loc[mask_uji].copy()
#
# X_uji = df_uji_test[fitur_ada]
# y_uji = df_uji_test['label'].values
#
# print(f'Blok uji: {len(df_uji_test)} sesi')
# print(f'Distribusi: {pd.Series(y_uji).value_counts().to_dict()}')
#
# # Baseline
# m_maj_test = baseline_mayoritas(y_uji)
# cetak_metrik('Baseline: Mayoritas (test)', m_maj_test)
#
# m_ruang_test = baseline_rata_historis_ruang(y_uji, df_uji_test['room_id'].values, rata_ruang)
# cetak_metrik('Baseline: Rata-rata per ruang (test)', m_ruang_test)
#
# m_jenis_test = baseline_rata_historis_jenis_hari(y_uji, df_uji_test['jenis_hari'].values, rata_jenis)
# cetak_metrik('Baseline: Rata-rata per jenis hari (test)', m_jenis_test)
#
# # Model + bootstrap CI
# for nama_model, key in [('Decision Tree', 'tree'), ('Logistic Regression', 'logreg')]:
#     pipe = hasil[key]['pipeline']
#     ambang = hasil[key]['ambang']
#     y_proba = pipe.predict_proba(X_uji)[:, 1]
#     y_pred_opt = (y_proba >= ambang).astype(int)
#     m = hitung_metrik(y_uji, y_pred_opt, y_proba)
#     ci = bootstrap_ci(y_uji, y_pred_opt, y_proba)
#     cetak_metrik(f'{nama_model} (ambang={ambang}, test)', m, ci)
#
#     y_pred_05 = (y_proba >= 0.5).astype(int)
#     m_05 = hitung_metrik(y_uji, y_pred_05, y_proba)
#     cetak_metrik(f'{nama_model} (ambang=0.5, test)', m_05)
#
# # Irisan
# pipe_lr_test = hasil['logreg']['pipeline']
# ambang_lr_test = hasil['logreg']['ambang']
# y_proba_lr_test = pipe_lr_test.predict_proba(X_uji)[:, 1]
# y_pred_lr_test = (y_proba_lr_test >= ambang_lr_test).astype(int)
# cetak_irisan_uji(df_uji_test, y_uji, y_pred_lr_test, y_proba_lr_test, cal_exc_test)
#
# session_test.close()
# print('\n✓ Evaluasi akhir selesai. JANGAN mengubah model setelah ini.')

In [ ]:
session_dev.close()
print('Session ditutup.')